# Groundwater Analysis Report (GAR)

In [ ]:
import matplotlib.pyplot as plt
import brodata

We can download the data of one Groundwater Analysis Report, using the class method `from_bro_id`. This returns a GroundwaterAnalysisReport-object that contains all data from the downloaded xml-file.

In [ ]:
gar = brodata.gar.GroundwaterAnalysisReport.from_bro_id("GAR000000019636")
gar

We can look at the contents of this GroundwaterAnalysisReport by using the `to_dict()` method.

In [ ]:
gar.to_dict()

The measurements are contained in the attribute `laboratoryAnalysis`, which is a pandas DataFrame.

In [ ]:
gar.laboratoryAnalysis

## Multiple objects

### All measurements of one tube

In [ ]:
gmw_id = "GMW000000017707"
tube_number = 1
df = brodata.gmw.get_tube_observations(gmw_id, tube_number, kind="gar")
df

In [ ]:
parameter_list = brodata.gar.get_parameter_list()
parameter_list

In [ ]:
# add the parameter description
df["parameter_description"] = parameter_list.loc[df["parameter"], "description"].values

In [ ]:
# show all the unique parameters and the number of measurements
parameter_description_counts = df["parameter_description"].value_counts()
parameter_description_counts

In [ ]:
parameter_description = "sulfaat"
parameter_code = brodata.gar.get_parameter_code(parameter_description, parameter_list=parameter_list)
ax = df.loc[parameter_code, "analysisMeasurementValue"].plot(marker=".")
uom = df.loc[parameter_code, "uom"].unique()
assert len(uom) == 1
ax.set_ylabel(uom[0])
ax.set_title(f"{parameter_description} {gmw_id}_{tube_number}");

### All measurements within extent
Just like with Groundwater Level Dossiers, there are two methods to download gar-data within an extent (`brodata.gm.get_data_in_extent` and `brodata.gmw.get_data_in_extent`). Both methods return a geopandas GeoDataFrame. The resulting measurement-data is contained in the column `laboratoryAnalysis`, and the bro-ids of the GroundwaterAnalysisReports of each tube are contained in the column `groundwaterAnalysisReport`.

Because of the greater simplicity, explained in the section about the Groundwater Level Dossiers, we wil use the gm-method (`brodata.gm.get_data_in_extent`). 

In [ ]:
extent = [115000, 120000, 438000, 441000]
gdf = brodata.gm.get_data_in_extent(extent=extent, kind="gar", combine=True)

In [ ]:
gdf.T

We can also plot the resulting GeoDataFrame on a map. We then see both Groundwater Monitoring Wells are located at the same location. 

In [ ]:
f, ax = plt.subplots()
ax.axis("scaled")
ax.axis(extent)
gdf.plot(ax=ax);

We can then also plot the measurement-series for these measurement tubes in one plot.

In [ ]:
f, ax = plt.subplots()
parameter_description = "sulfaat"
parameter_code = brodata.gar.get_parameter_code(parameter_description, parameter_list=parameter_list)
uom = "mg/l"
for index in gdf.index:
    df = gdf.at[index, "laboratoryAnalysis"]
    mask = df["parameter"] == parameter_code
    if not mask.any():
        continue
    assert (df.loc[mask, "uom"] == uom).all()
    ax = df.loc[mask, "analysisMeasurementValue"].plot(marker=".", ax=ax, label=index)
ax.set_ylabel(uom)
ax.set_title(parameter_description)
ax.legend();